In [1]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/pseudobulk_test/yao_2021/ACA")

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/enrichment_fxns.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/gene_mapping_fxns.R")
# source("/mnt/lareaulab/reliscu/code/upper_first.R")

library(dplyr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last




Here I perform enrichment analysis to find modules enriched for cell type markers. 

These modules will later be used to correlate with exon PSI to define cell type-specific exons.

In [2]:
mod_def <- "PosBC"
unique <- FALSE

## Prep DE genes

### ACA pseudobulk DE genes

In [50]:
set_source <- "yao_2021_ACA_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes"

pairwise_res_list <- readRDS("data/DE_genes/yao_2021_ACA_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes_dream.RDS")
marker_genes_list <- prep_DE_genes(pairwise_res_list, pairwise=TRUE, unique=unique)

In [52]:
names(marker_genes_list)

[1] "Endo"        "L2_3_IT_CTX" "L2_3_IT_PPP" "L4_5_IT_CTX" "L5_6_NP_CTX"
 [6] "L5_IT_CTX"   "L5_PT_CTX"   "L6_CT_CTX"   "L6_IT_CTX"   "L6b_CTX"    
[11] "Lamp5"       "Oligo"       "Pvalb"       "Sncg"        "Sst_Chodl"  
[16] "Vip"

### MOp pseudobulk DE genes

In [3]:
set_source <- "yao_2021_MOp_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes"

pairwise_res_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/pseudobulk_test/yao_2021/MOp/data/DE_genes/yao_2021_MOp_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes_dream.RDS")
marker_genes_list <- prep_DE_genes(pairwise_res_list, pairwise=TRUE, unique=unique)
names(marker_genes_list) <- unlist(sapply(names(marker_genes_list), function(x) gsub(" ", "_", x)))

In [4]:
names(marker_genes_list)

[1] "Astro"       "Car3"        "L2_3_IT_CTX" "L4_5_IT_CTX" "L5_6_NP_CTX"
 [6] "L5_IT_CTX"   "L5_PT_CTX"   "L6_CT_CTX"   "L6_IT_CTX"   "L6b_CTX"    
[11] "Lamp5"       "Pvalb"       "SMC-Peri"    "Sncg"        "Sst_Chodl"  
[16] "Vip"

### Claude marker genes

In [10]:
set_source <- "Claude_marker_genes"

marker_genes_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/marker_genes/AI/Claude_cortical_markers_mouse.RDS")
names(marker_genes_list) <- sapply(names(marker_genes_list), function(x) gsub(" ", "_", gsub("/", "_", x, fixed=TRUE)))

### Gugene DE genes

In [ ]:
# marker_genes_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/kang_2026_preprint/kang_2026_preprint_Jorstad_2023_DE_genes.RData")
# marker_genes_list <- lapply(marker_genes_list, convert_genes)

# marker_genes_list1 <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/kang_2026_preprint/kang_2026_preprint_Jorstad_2023_DE_genes.RData")
# marker_genes_list2 <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/kang_2026_preprint/kang_2026_preprint_Gabitto_2024_DE_genes.RData")

# marker_genes_list <- lapply(seq_along(marker_genes_list1), function(i) {
#     union(marker_genes_list1[[i]], marker_genes_list2[[i]])
# })
# marker_genes_list <- lapply(marker_genes_list, convert_genes)

## Round 1 (cell_type, pcntVar = 20, mergeParam = 0.85)

In [4]:
network_dir <- "yao_2021_ACA_STAR_20pcntCells_25SD_200samples_pseudobulk_mergeParam0.85_Modules"
dataset <- "20pcntCells_25SD_200samples_pseudobulk_mergeParam0.85"

In [ ]:
enrichments_df <- get_module_enrichments(network_dir, marker_genes_list, mod_def)

In [ ]:
# Get most enriched cell type for each module
# If cell type is most enriched in multiple modules, choose module with smallest p-value

top_mods_df <- enrichments_df %>%
    group_by(Cell_type) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    group_by(Network, Module) %>%
    arrange(Network) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    arrange(Qval)

In [ ]:
top_mods_df[,c("Cell_type", "Pval", "Qval", "Module", "Network")]

Cell_type,Pval,Qval,Module,Network
<chr>,<dbl>,<dbl>,<chr>,<chr>
endothelial_cell,0.000000e+00,0.000000e+00,turquoise,Bicor-None_signum0.121_minSize10_merge_ME_0.85_19015
oligodendrocyte,2.166937e-245,3.533801e-242,blue,Bicor-None_signum0.324_minSize8_merge_ME_0.85_19015
pvalb_GABAergic_cortical_interneuron,1.892308e-158,1.567462e-155,brown,Bicor-None_signum0.41_minSize10_merge_ME_0.85_19015
L4_5_intratelencephalic_projecting_glutamatergic_neuron_of_the_primary_motor_cortex,3.973249e-156,3.239750e-153,purple,Bicor-None_signum0.277_minSize8_merge_ME_0.85_19015
L5_6_near-projecting_glutamatergic_neuron_of_the_primary_motor_cortex,3.199823e-134,2.140804e-131,magenta,Bicor-None_signum0.277_minSize10_merge_ME_0.85_19015
sst_chodl_GABAergic_cortical_interneuron,6.364315e-109,2.863119e-106,pink,Bicor-None_signum0.674_minSize12_merge_ME_0.85_19015
L6_corticothalamic-projecting_glutamatergic_cortical_neuron,2.437433e-96,1.042602e-93,tan,Bicor-None_signum0.674_minSize10_merge_ME_0.85_19015
VIP_GABAergic_cortical_interneuron,1.971001e-89,7.973385e-87,tan,Bicor-None_signum0.245_minSize15_merge_ME_0.85_19015
lamp5_GABAergic_cortical_interneuron,2.094271e-85,7.806394e-83,cyan,Bicor-None_signum0.41_minSize12_merge_ME_0.85_19015


In [ ]:
write.csv(top_mods_df, file=paste0("data/enrichments/yao_2021_ACA_pairwise_DE_genes_dream_unique", unique, "_", dataset, "_", mod_def, "_top_Qval_modules.csv"), row.names=FALSE, quote=FALSE)

## Round 2 (cell_type, pcntVar = 20, mergeParam = 0.9)

In [ ]:
network_dir <- "yao_2021_ACA_STAR_20pcntCells_20pcntVar_200samples_pseudobulk_mergeParam0.9_Modules"
dataset <- "20pcntCells_20pcntVar_200samples_pseudobulk_mergeParam0.9"

In [ ]:
enrichments_df <- get_module_enrichments(network_dir, marker_genes_list, mod_def)

In [ ]:
# Get most enriched cell type for each module
# If cell type is most enriched in multiple modules, choose module with smallest p-value

top_mods_df <- enrichments_df %>%
    group_by(Cell_type) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    group_by(Network, Module) %>%
    arrange(Network) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    arrange(Qval)

In [ ]:
top_mods_df[,c("Cell_type", "Pval", "Qval", "Module", "Network")]

Cell_type,Pval,Qval,Module,Network
<chr>,<dbl>,<dbl>,<chr>,<chr>
L4_5_intratelencephalic_projecting_glutamatergic_neuron_of_the_primary_motor_cortex,4.867242e-133,1.984043e-128,blue,Bicor-None_signum0.341_minSize12_merge_ME_0.9_19017
endothelial_cell,5.183388e-113,1.039269e-108,black,Bicor-None_signum0.205_minSize15_merge_ME_0.9_19017
L6_corticothalamic-projecting_glutamatergic_cortical_neuron,1.372387e-72,2.358545e-69,turquoise,Bicor-None_signum0.341_minSize15_merge_ME_0.9_19017
oligodendrocyte,6.760142e-68,1.070059e-64,brown,Bicor-None_signum0.341_minSize12_merge_ME_0.9_19017
L5_6_near-projecting_glutamatergic_neuron_of_the_primary_motor_cortex,3.526575e-67,5.303087e-64,darkmagenta,Bicor-None_signum0.341_minSize8_merge_ME_0.9_19017
pvalb_GABAergic_cortical_interneuron,5.709826e-53,3.150881e-50,yellow,Bicor-None_signum0.341_minSize12_merge_ME_0.9_19017
L6b_glutamatergic_cortical_neuron,1.143973e-47,5.135071e-45,skyblue,Bicor-None_signum0.205_minSize12_merge_ME_0.9_19017
VIP_GABAergic_cortical_interneuron,2.335827e-42,8.836479e-40,sienna3,Bicor-None_signum0.205_minSize12_merge_ME_0.9_19017
L6_intratelencephalic_projecting_glutamatergic_neuron_of_the_primary_motor_cortex,3.698239e-39,1.195963e-36,lightgreen,Bicor-None_signum0.0858_minSize15_merge_ME_0.9_19017


In [ ]:
write.csv(top_mods_df, file=paste0("data/enrichments/", network_dir, "_", set_source, "_Qval_modules.csv"), row.names=FALSE, quote=FALSE)

## Round 3 (cell_type, SD = 25, mergeParam = 0.85)

In [114]:
network_dir <- "yao_2021_ACA_STAR_SyntheticDataset1_cell_type_20pcntCells_25SD_200samples_pseudobulk_mergeParam0.85_Modules"

In [115]:
enrichments_df <- get_module_enrichments(network_dir, marker_genes_list, mod_def)

In [116]:
# Get most enriched cell type for each module
# If cell type is most enriched in multiple modules, choose module with smallest p-value

top_mods_df <- enrichments_df %>%
    group_by(Cell_type) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    group_by(Network, Module) %>%
    arrange(Network) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    arrange(Qval)

In [117]:
top_mods_df[,c("Cell_type", "Pval", "Qval", "Module", "Network")]

Cell_type,Pval,Qval,Module,Network
<chr>,<dbl>,<dbl>,<chr>,<chr>
endothelial_cell,0.000000e+00,0.000000e+00,turquoise,Bicor-None_signum0.121_minSize10_merge_ME_0.85_19015
oligodendrocyte,2.166937e-245,3.533801e-242,blue,Bicor-None_signum0.324_minSize8_merge_ME_0.85_19015
pvalb_GABAergic_cortical_interneuron,1.892308e-158,1.567462e-155,brown,Bicor-None_signum0.41_minSize10_merge_ME_0.85_19015
L4_5_intratelencephalic_projecting_glutamatergic_neuron_of_the_primary_motor_cortex,3.973249e-156,3.239750e-153,purple,Bicor-None_signum0.277_minSize8_merge_ME_0.85_19015
L5_6_near-projecting_glutamatergic_neuron_of_the_primary_motor_cortex,3.199823e-134,2.140804e-131,magenta,Bicor-None_signum0.277_minSize10_merge_ME_0.85_19015
sst_chodl_GABAergic_cortical_interneuron,6.364315e-109,2.863119e-106,pink,Bicor-None_signum0.674_minSize12_merge_ME_0.85_19015
L6_corticothalamic-projecting_glutamatergic_cortical_neuron,2.437433e-96,1.042602e-93,tan,Bicor-None_signum0.674_minSize10_merge_ME_0.85_19015
VIP_GABAergic_cortical_interneuron,1.971001e-89,7.973385e-87,tan,Bicor-None_signum0.245_minSize15_merge_ME_0.85_19015
lamp5_GABAergic_cortical_interneuron,2.094271e-85,7.806394e-83,cyan,Bicor-None_signum0.41_minSize12_merge_ME_0.85_19015


In [107]:
top_mods_df[,c("Cell_type", "Pval", "Qval", "Module", "Network")]

Cell_type,Pval,Qval,Module,Network
<chr>,<dbl>,<dbl>,<chr>,<chr>
Endo,4.014688e-23,4.441851e-18,turquoise,Bicor-None_signum0.22_minSize6_merge_ME_0.95_26598
Oligo,3.000326e-08,1.508891e-04,cyan,Bicor-None_signum0.783_minSize3_merge_ME_0.95_26598
OPC,9.098239e-07,3.728256e-03,brown,Bicor-None_signum0.333_minSize6_merge_ME_0.95_26598
Micro/PVM,7.925762e-06,2.307648e-02,black,Bicor-None_signum0.258_minSize10_merge_ME_0.95_26598
Pax6,1.093314e-05,3.024106e-02,darkmagenta,Bicor-None_signum0.597_minSize6_merge_ME_0.95_26598
VLMC,2.530937e-05,5.490644e-02,black,Bicor-None_signum0.22_minSize6_merge_ME_0.95_26598
L2/3 IT,2.447831e-05,5.490644e-02,magenta1,Bicor-None_signum0.597_minSize4_merge_ME_0.95_26598
Astro,7.573647e-05,1.341814e-01,orangered1,Bicor-None_signum0.258_minSize8_merge_ME_0.95_26598
L4 IT,7.761761e-05,1.341814e-01,plum4,Bicor-None_signum0.783_minSize3_merge_ME_0.95_26598


In [ ]:
write.csv(top_mods_df, file=paste0("data/enrichments/", network_dir, "_", set_source, "_Qval_modules.csv"), row.names=FALSE, quote=FALSE)

## Round 4 (subclass_label, SD = 25, mergeParam = 0.95)

In [11]:
network_dir <- "yao_2021_ACA_STAR_SyntheticDataset1_subclass_label_20pcntCells_25SD_200samples_mergeParam0.95_subsetCutoff8.24_Modules"

In [12]:
enrichments_df <- get_module_enrichments(network_dir, marker_genes_list, mod_def)

In [16]:
# Get most enriched cell type for each module
# If cell type is most enriched in multiple modules, choose module with smallest p-value

top_mods_df <- enrichments_df %>%
    group_by(Cell_type) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    group_by(Network, Module) %>%
    arrange(Network) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    arrange(Qval)

In [17]:
top_mods_df[,c("Cell_type", "Pval", "Qval", "Module", "Network")]

Cell_type,Pval,Qval,Module,Network
<chr>,<dbl>,<dbl>,<chr>,<chr>
Oligo,2.014753e-25,5.805007e-21,blue,Bicor-None_signum0.22_minSize10_merge_ME_0.95_26598
Endo,1.061499e-22,8.738411e-19,turquoise,Bicor-None_signum0.597_minSize3_merge_ME_0.95_26598
Sncg,1.021633e-18,3.364093e-15,greenyellow,Bicor-None_signum0.783_minSize6_merge_ME_0.95_26598
Pvalb,1.202202e-15,3.222180e-12,pink,Bicor-None_signum0.783_minSize10_merge_ME_0.95_26598
Sst,2.629672e-15,6.588470e-12,magenta,Bicor-None_signum0.783_minSize8_merge_ME_0.95_26598
Micro_PVM,2.356705e-13,5.223275e-10,ghostwhite,Bicor-None_signum0.258_minSize6_merge_ME_0.95_26598
OPC,1.067019e-12,2.195963e-09,blue,Bicor-None_signum0.22_minSize6_merge_ME_0.95_26598
L6b,2.967213e-12,5.896056e-09,darkolivegreen,Bicor-None_signum0.333_minSize6_merge_ME_0.95_26598
L6_IT_Car3,1.340245e-10,2.175539e-07,darkred,Bicor-None_signum0.783_minSize6_merge_ME_0.95_26598


In [18]:
write.csv(top_mods_df, file=paste0("data/enrichments/", network_dir, "_", set_source, "_enrichments.csv"), row.names=FALSE, quote=FALSE)